# LeetCode 319 — Bulb Switcher bằng Python

Link đề: https://leetcode.com/problems/bulb-switcher/ · Độ khó: **Medium** · Chủ đề: Math, Brainteaser

Skeleton LeetCode:

```python
class Solution:
    def bulbSwitch(self, n: int) -> int:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` bóng đèn ban đầu **tắt**; vòng 1 bật tất cả; vòng 2 đổi trạng thái các bóng ở vị trí chia hết cho 2; vòng `k` đổi trạng thái các bóng ở vị trí chia hết cho `k`; ... đến vòng `n`. Đếm số bóng **đang sáng** sau vòng `n`.
2. Đi từ **mô phỏng trực tiếp `O(n²)`** → **đếm ước số (bóng `i` bị đổi trạng thái đúng `d(i)` lần)** → **nhận ra bóng sáng ⇔ số ước số lẻ ⇔ `i` là số chính phương** → **đếm số chính phương `≤ n` bằng vòng lặp `O(√n)`** → **công thức `⌊√n⌋` (`math.isqrt`)**, cùng **tìm kiếm nhị phân** và **Newton** cho căn bậc hai nguyên.
3. Hiểu **vì sao ước số đi theo cặp** `(d, i/d)` và chỉ số chính phương mới có một ước "tự ghép cặp".
4. Biết **sai số của căn bậc hai dấu phẩy động** và vì sao `math.isqrt` đáng tin hơn `int(n ** 0.5)` với số lớn.
5. Tránh các lỗi kinh điển: quên vòng 1, làm tròn lên thay vì xuống, `<` thay `<=` khi đếm, dùng số thực với `n` quá lớn.

## 2. Đề bài & ví dụ

Có `n` bóng đèn, ban đầu đều **tắt**. Bạn thực hiện `n` vòng:

- Vòng 1: bật tất cả các bóng.
- Vòng 2: đổi trạng thái (bật ↔ tắt) mỗi bóng thứ 2 (bóng 2, 4, 6, ...).
- Vòng 3: đổi trạng thái mỗi bóng thứ 3 (bóng 3, 6, 9, ...).
- ...
- Vòng `i`: đổi trạng thái mỗi bóng thứ `i`.
- Vòng `n`: chỉ đổi trạng thái bóng thứ `n`.

Trả về **số bóng đang sáng sau `n` vòng**.

**Ví dụ 1:** `n = 3` → `1`. Ban đầu `[off, off, off]`; sau vòng 1 `[on, on, on]`; sau vòng 2 `[on, off, on]`; sau vòng 3 `[on, off, off]`. Còn một bóng sáng.

**Ví dụ 2:** `n = 0` → `0`.

**Ví dụ 3:** `n = 1` → `1`.

**Ràng buộc:** `0 <= n <= 10⁹`.

In [1]:
import math
import random

examples = [(3, 1), (0, 0), (1, 1)]
for n, e in examples:
    print("n =", n, "-> kỳ vọng", e)

n = 3 -> kỳ vọng 1
n = 0 -> kỳ vọng 0
n = 1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Bóng `i` bị đổi trạng thái bao nhiêu lần?

Bóng `i` bị đổi ở vòng `k` đúng khi `k` là **ước số** của `i` (`k | i`). Vậy bóng `i` bị đổi trạng thái đúng `d(i)` lần, với `d(i)` là **số ước số dương** của `i` (kể cả `1` và `i`). Vì ban đầu tắt, bóng `i` cuối cùng **sáng ⇔ `d(i)` lẻ**.

In [2]:
def divisors(i):
    return [k for k in range(1, i + 1) if i % k == 0]


for i in range(1, 11):
    ds = divisors(i)
    print(f"bóng {i:2}: ước = {ds}  -> {len(ds)} lần đổi -> {'SÁNG' if len(ds) % 2 else 'tắt'}")

bóng  1: ước = [1]  -> 1 lần đổi -> SÁNG
bóng  2: ước = [1, 2]  -> 2 lần đổi -> tắt
bóng  3: ước = [1, 3]  -> 2 lần đổi -> tắt
bóng  4: ước = [1, 2, 4]  -> 3 lần đổi -> SÁNG
bóng  5: ước = [1, 5]  -> 2 lần đổi -> tắt
bóng  6: ước = [1, 2, 3, 6]  -> 4 lần đổi -> tắt
bóng  7: ước = [1, 7]  -> 2 lần đổi -> tắt
bóng  8: ước = [1, 2, 4, 8]  -> 4 lần đổi -> tắt
bóng  9: ước = [1, 3, 9]  -> 3 lần đổi -> SÁNG
bóng 10: ước = [1, 2, 5, 10]  -> 4 lần đổi -> tắt


### 3.2. Ước số đi theo cặp

Mỗi ước `d` của `i` có "bạn đồng hành" `i / d`; hai ước này tạo thành một cặp `(d, i/d)`. Các cặp **khác nhau** thì mỗi cặp góp **2** ước. Số ước chỉ có thể **lẻ** khi có một ước ghép với **chính nó**: `d = i / d`, tức `i = d²`: `i` là **số chính phương**. Ngược lại, mọi số chính phương `d²` có đúng một ước tự ghép (`d`), nên số ước là lẻ.

In [3]:
for i in (12, 16, 36):
    pairs = [(d, i // d) for d in range(1, int(i ** 0.5) + 1) if i % d == 0]
    print(f"{i}: các cặp {pairs} -> {'có cặp tự ghép' if any(a == b for a, b in pairs) else 'không có cặp tự ghép'}")

12: các cặp [(1, 12), (2, 6), (3, 4)] -> không có cặp tự ghép
16: các cặp [(1, 16), (2, 8), (4, 4)] -> có cặp tự ghép
36: các cặp [(1, 36), (2, 18), (3, 12), (4, 9), (6, 6)] -> có cặp tự ghép


### 3.3. Căn bậc hai nguyên: `⌊√n⌋`

Số chính phương trong `[1, n]` là `1², 2², ..., k²` với `k = ⌊√n⌋`. Trong Python dùng `math.isqrt(n)` (chính xác cho số nguyên lớn tùy ý). `int(math.sqrt(n))` dùng số thực 53 bit nên có thể sai với `n` rất lớn (xem phần lỗi).

In [4]:
for n in (0, 1, 3, 4, 8, 9, 10, 15, 16, 99, 100):
    print(f"n = {n:3}: isqrt = {math.isqrt(n)}, k*k <= n < (k+1)^2: {math.isqrt(n) ** 2} <= {n} < {(math.isqrt(n) + 1) ** 2}")

n =   0: isqrt = 0, k*k <= n < (k+1)^2: 0 <= 0 < 1
n =   1: isqrt = 1, k*k <= n < (k+1)^2: 1 <= 1 < 4
n =   3: isqrt = 1, k*k <= n < (k+1)^2: 1 <= 3 < 4
n =   4: isqrt = 2, k*k <= n < (k+1)^2: 4 <= 4 < 9
n =   8: isqrt = 2, k*k <= n < (k+1)^2: 4 <= 8 < 9
n =   9: isqrt = 3, k*k <= n < (k+1)^2: 9 <= 9 < 16
n =  10: isqrt = 3, k*k <= n < (k+1)^2: 9 <= 10 < 16
n =  15: isqrt = 3, k*k <= n < (k+1)^2: 9 <= 15 < 16
n =  16: isqrt = 4, k*k <= n < (k+1)^2: 16 <= 16 < 25
n =  99: isqrt = 9, k*k <= n < (k+1)^2: 81 <= 99 < 100
n = 100: isqrt = 10, k*k <= n < (k+1)^2: 100 <= 100 < 121


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`n = 10`. Số lần đổi trạng thái của mỗi bóng: `d(1) = 1`, `d(2) = 2`, `d(3) = 2`, `d(4) = 3`, `d(5) = 2`, `d(6) = 4`, `d(7) = 2`, `d(8) = 4`, `d(9) = 3`, `d(10) = 4`.

Số lẻ: bóng `1, 4, 9` (đều là số chính phương). Vậy đáp án `3 = ⌊√10⌋`.

Cách nhìn nhanh: chỉ những bóng **số chính phương** mới sáng, và số chính phương không vượt quá `n` là `⌊√n⌋`.

In [5]:
n = 10
state = [False] * (n + 1)
for k in range(1, n + 1):
    for i in range(k, n + 1, k):
        state[i] = not state[i]
print("bóng sáng:", [i for i in range(1, n + 1) if state[i]], "-> đếm", sum(state[1:]))

bóng sáng: [1, 4, 9] -> đếm 3


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng như đề

**Ý tưởng.** Mảng trạng thái `n` bóng; với mỗi vòng `k` đổi trạng thái các bóng bội của `k`; cuối cùng đếm bóng sáng. Tổng số lần đổi `n/1 + n/2 + ... + n/n = n·H_n ≈ n ln n`.

In [6]:
def bulb_switch_simulate(n: int) -> int:
    on = [False] * (n + 1)
    for k in range(1, n + 1):
        for i in range(k, n + 1, k):
            on[i] = not on[i]
    return sum(on)


for n, e in examples:
    assert bulb_switch_simulate(n) == e
print("PASS ví dụ (mô phỏng)")

PASS ví dụ (mô phỏng)


**Độ phức tạp (Cách 1).** Time `O(n log n)` (tổng điều hòa `n·H_n`); Space `O(n)`. Với `n = 10⁹` là **không khả thi** (hàng chục tỉ thao tác, 1 GB bộ nhớ). Dùng làm **oracle** cho `n` nhỏ.

**Nút thắt.** Ta chỉ cần biết số lần mỗi bóng bị đổi là chẵn hay lẻ, không cần mô phỏng từng lần.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Chẵn/lẻ thay vì từng lần đổi:** bóng `i` sáng ⇔ `d(i)` lẻ.
2. **Ước đi theo cặp ⇒ `d(i)` lẻ ⇔ `i` là số chính phương.**
3. **Đáp án = số chính phương trong `[1, n]` = `⌊√n⌋`.**

Bước 3 cho lời giải `O(1)` (hoặc `O(log n)` nếu tự tính căn bậc hai nguyên). Đây là một bài **toán đố (brainteaser)**: chìa khóa là *quan sát toán học*, không phải thuật toán phức tạp.

In [7]:
# Kiểm chứng định lý: bóng i sáng  <=>  i là số chính phương  (với mọi n <= 400)
for n in range(1, 401):
    on = [False] * (n + 1)
    for k in range(1, n + 1):
        for i in range(k, n + 1, k):
            on[i] = not on[i]
    lit = [i for i in range(1, n + 1) if on[i]]
    assert lit == [r * r for r in range(1, math.isqrt(n) + 1)]
print("PASS: với mọi n <= 400, tập bóng sáng chính là các số chính phương <= n")

PASS: với mọi n <= 400, tập bóng sáng chính là các số chính phương <= n


## 7. Các cách tiếp theo

### Cách 2 — Đếm ước số từng bóng (không mô phỏng)

**Giải thích.** Với mỗi `i ∈ [1, n]` đếm số ước (duyệt `d` đến `√i` và đếm theo cặp), đếm các `i` có số ước lẻ. Time `O(n√n)`; chỉ dùng cho `n` vừa. Dùng làm oracle thứ hai, độc lập với mô phỏng.

In [8]:
def count_divisors(i: int) -> int:
    cnt = 0
    d = 1
    while d * d <= i:
        if i % d == 0:
            cnt += 1 if d * d == i else 2           # ước d và i/d (một ước nếu d = i/d)
        d += 1
    return cnt


def bulb_switch_divisors(n: int) -> int:
    return sum(1 for i in range(1, n + 1) if count_divisors(i) % 2 == 1)


for n, e in examples:
    assert bulb_switch_divisors(n) == e
print("PASS ví dụ (đếm ước số)")

PASS ví dụ (đếm ước số)


**Độ phức tạp Cách 2:** Time `O(n√n)`; Space `O(1)`.

### Cách 3 — Đếm số chính phương bằng vòng lặp `k·k ≤ n`

**Giải thích.** Biết đáp án là số chính phương, ta đếm `k = 1, 2, ...` cho đến khi `k·k > n`. Time `O(√n)` (với `n = 10⁹` chỉ khoảng 31 623 bước).

In [9]:
def bulb_switch_loop(n: int) -> int:
    k = 0
    while (k + 1) * (k + 1) <= n:
        k += 1
    return k


for n, e in examples:
    assert bulb_switch_loop(n) == e
print("PASS ví dụ (vòng lặp)")

PASS ví dụ (vòng lặp)


**Độ phức tạp Cách 3:** Time `O(√n)`; Space `O(1)`.

### Cách 4 — Công thức `⌊√n⌋` với `math.isqrt` (cách nộp bài)

**Giải thích.** `math.isqrt(n)` trả `⌊√n⌋` **chính xác** cho số nguyên không âm. Một dòng, `O(1)` về mặt "số thao tác" (thực tế `O(log n)` bit).

In [10]:
def bulb_switch_isqrt(n: int) -> int:
    return math.isqrt(n)


for n, e in examples:
    assert bulb_switch_isqrt(n) == e
print("PASS ví dụ (isqrt)")

PASS ví dụ (isqrt)


**Độ phức tạp Cách 4:** Time `O(1)` (hằng số nhỏ; `isqrt` thực hiện `O(log n)` bước trên số nguyên); Space `O(1)`.

### Cách 5 — Tự tính căn bậc hai nguyên: tìm kiếm nhị phân và Newton

**Giải thích.** Không dùng thư viện: tìm `k` lớn nhất sao cho `k² ≤ n` bằng tìm kiếm nhị phân trên `[0, n]` (`O(log n)`), hoặc bằng phương pháp Newton `x ← (x + n // x) // 2` (hội tụ nhanh, `O(log log n)` bước số học nhưng mỗi bước chia nguyên).

In [11]:
def bulb_switch_bsearch(n: int) -> int:
    lo, hi = 0, n                                   # tìm k lớn nhất với k*k <= n
    while lo < hi:
        mid = (lo + hi + 1) // 2                    # chọn mid lệch lên để tránh lặp vô hạn
        if mid * mid <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo


def bulb_switch_newton(n: int) -> int:
    if n < 2:
        return n
    x = n
    while True:
        y = (x + n // x) // 2
        if y >= x:
            return x
        x = y


for n, e in examples:
    assert bulb_switch_bsearch(n) == e and bulb_switch_newton(n) == e
print("PASS ví dụ (nhị phân và Newton)")

PASS ví dụ (nhị phân và Newton)


**Độ phức tạp Cách 5:** Nhị phân `O(log n)`; Newton `O(log n)` trong trường hợp xấu (thực tế rất ít vòng); Space `O(1)`.

> Cách 4 là cách nộp bài. Cách 5 hữu ích khi không có `isqrt` (ví dụ ngôn ngữ chỉ có `sqrt` số thực). Cách 1 và 2 làm oracle.

## 8. TRACE từng bước

Mô phỏng `n = 10` từng vòng: in trạng thái sau mỗi vòng (`#` sáng, `.` tắt).

In [12]:
def trace_rounds(n):
    on = [False] * (n + 1)
    print("bóng :", " ".join(f"{i:2}" for i in range(1, n + 1)))
    for k in range(1, n + 1):
        for i in range(k, n + 1, k):
            on[i] = not on[i]
        print(f"vòng {k:2}:", " ".join(" #" if on[i] else " ." for i in range(1, n + 1)))
    print("bóng sáng:", [i for i in range(1, n + 1) if on[i]])
    return sum(on)


assert trace_rounds(10) == 3

bóng :  1  2  3  4  5  6  7  8  9 10
vòng  1:  #  #  #  #  #  #  #  #  #  #
vòng  2:  #  .  #  .  #  .  #  .  #  .
vòng  3:  #  .  .  .  #  #  #  .  .  .
vòng  4:  #  .  .  #  #  #  #  #  .  .
vòng  5:  #  .  .  #  .  #  #  #  .  #
vòng  6:  #  .  .  #  .  .  #  #  .  #
vòng  7:  #  .  .  #  .  .  .  #  .  #
vòng  8:  #  .  .  #  .  .  .  .  .  #
vòng  9:  #  .  .  #  .  .  .  .  #  #
vòng 10:  #  .  .  #  .  .  .  .  #  .
bóng sáng: [1, 4, 9]


Sau vòng 10 chỉ còn sáng các bóng `1, 4, 9`.

| bóng | ước số | số lần đổi | trạng thái cuối |
|---|---|---|---|
| 1 | 1 | 1 | sáng |
| 2 | 1, 2 | 2 | tắt |
| 4 | 1, 2, 4 | 3 | **sáng** |
| 6 | 1, 2, 3, 6 | 4 | tắt |
| 9 | 1, 3, 9 | 3 | **sáng** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (`math.isqrt`). Nếu muốn viết không phụ thuộc thư viện: `int(n ** 0.5)` đủ chính xác với `n ≤ 10⁹`, nhưng `isqrt` luôn an toàn.

In [13]:
import math


class Solution:
    def bulbSwitch(self, n: int) -> int:
        return math.isqrt(n)


for n, e in examples:
    assert Solution().bulbSwitch(n) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Bóng `i` bị đổi ở vòng `k` ⇔ `k | i`** (vòng `k` chỉ đụng tới các bội của `k`). Số lần đổi là số ước `d(i)`.
- **Trạng thái cuối:** ban đầu tắt; mỗi lần đổi đảo trạng thái; sau `d(i)` lần đổi, bóng sáng ⇔ `d(i)` lẻ.
- **Ước ghép cặp:** ánh xạ `d ↦ i / d` là một phép đảo ngược (involution) trên tập ước số. Điểm bất động `d = i / d` ⇔ `d² = i`. Nếu `i` không phải chính phương thì không có điểm bất động, các ước chia thành các cặp `{d, i/d}` rời nhau ⇒ `d(i)` **chẵn**. Nếu `i = r²` thì có đúng một điểm bất động `d = r` (vì `d > 0` và `d² = i` có nghiệm duy nhất) ⇒ `d(i)` **lẻ**.
- **Đếm:** các số chính phương trong `[1, n]` là `1², ..., k²` với `k = ⌊√n⌋`, nên đáp án là `k`.
- **Biên:** `n = 0` ⇒ `0` bóng ⇒ `isqrt(0) = 0`; `n = 1` ⇒ bóng 1 sáng ⇒ `isqrt(1) = 1`.

## 11. Phân tích Time / Space Complexity

- **Cách 1 (mô phỏng):** Time `O(n log n)`; Space `O(n)`.
- **Cách 2 (đếm ước):** Time `O(n√n)`; Space `O(1)`.
- **Cách 3 (vòng lặp):** Time `O(√n)`; Space `O(1)`.
- **Cách 4 (`isqrt`):** Time `O(1)` (một lời gọi thư viện); Space `O(1)`.
- **Cách 5 (nhị phân / Newton):** Time `O(log n)`; Space `O(1)`.
- Với `n = 10⁹`: Cách 1 không chạy nổi; Cách 3 khoảng `3·10⁴` bước; Cách 4 và 5 tức thì.

## 12. Bộ test

Đối chiếu mọi cách với **mô phỏng** (cho `n` nhỏ) và với nhau (cho `n` lớn). Công thức `⌊√n⌋` cũng được kiểm tra bằng điều kiện trực tiếp `k² ≤ n < (k+1)²`.

In [14]:
impls = {
    "simulate": bulb_switch_simulate,
    "divisors": bulb_switch_divisors,
    "loop": bulb_switch_loop,
    "isqrt": bulb_switch_isqrt,
    "bsearch": bulb_switch_bsearch,
    "newton": bulb_switch_newton,
    "submit": lambda n: Solution().bulbSwitch(n),
}
SLOW = ("simulate", "divisors")


def check(n, expected=None):
    base = bulb_switch_simulate(n) if n <= 3000 else math.isqrt(n)
    if expected is not None:
        assert base == expected, (n, base, expected)
    assert base ** 2 <= n < (base + 1) ** 2
    for name, fn in impls.items():
        if name in SLOW and n > 3000:
            continue
        assert fn(n) == base, (name, n, fn(n), base)
    return base


for n, e in examples:
    check(n, e)
print("PASS 3 ví dụ của đề")

PASS 3 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [15]:
cases = [
    (0, 0, "không có bóng"),
    (1, 1, "một bóng"),
    (2, 1, "n = 2: chỉ bóng 1 sáng"),
    (3, 1, "ví dụ của đề"),
    (4, 2, "n = 4: bóng 1 và 4"),
    (8, 2, "ngay trước số chính phương 9"),
    (9, 3, "đúng số chính phương"),
    (10, 3, "n = 10"),
    (15, 3, "ngay trước 16"),
    (16, 4, "16"),
    (24, 4, "ngay trước 25"),
    (25, 5, "25"),
    (100, 10, "100"),
    (99, 9, "ngay trước 100"),
]
for n, e, why in cases:
    assert check(n, e) == e
    print("PASS", why)
for n, e in [(10 ** 9, 31622), (99999999, 9999), (999950884, 31622), (999950883, 31621), (1000000000 - 1, 31622)]:
    assert math.isqrt(n) == e
    for name in ("loop", "isqrt", "bsearch", "newton", "submit"):
        assert impls[name](n) == e, (name, n)
print("PASS các giá trị lớn nhất theo đề (n = 10^9 -> 31622, 31622^2 = 999950884)")

PASS không có bóng
PASS một bóng
PASS n = 2: chỉ bóng 1 sáng
PASS ví dụ của đề
PASS n = 4: bóng 1 và 4
PASS ngay trước số chính phương 9
PASS đúng số chính phương
PASS n = 10
PASS ngay trước 16
PASS 16
PASS ngay trước 25
PASS 25
PASS 100
PASS ngay trước 100
PASS các giá trị lớn nhất theo đề (n = 10^9 -> 31622, 31622^2 = 999950884)


### 12.2. Tính chất và phản ví dụ

In [16]:
rnd = random.Random(319)

# Tính chất 1: hàm không giảm theo n; tăng đúng 1 tại các số chính phương
prev = bulb_switch_isqrt(0)
for n in range(1, 5000):
    cur = bulb_switch_isqrt(n)
    assert cur - prev in (0, 1)
    assert (cur - prev == 1) == (math.isqrt(n) ** 2 == n)
    prev = cur
print("PASS tính chất: kết quả tăng đúng 1 tại mỗi số chính phương, các n khác giữ nguyên")

# Tính chất 2: số ước lẻ <=> chính phương
for i in range(1, 2000):
    assert (count_divisors(i) % 2 == 1) == (math.isqrt(i) ** 2 == i)
print("PASS tính chất: d(i) lẻ <=> i là số chính phương (i < 2000)")

# Phản ví dụ: sqrt dấu phẩy động sai với n rất lớn (ngoài ràng buộc của đề)
big = 10 ** 18 - 1
print("int(sqrt(10^18 - 1)) =", int(math.sqrt(big)), "| isqrt =", math.isqrt(big))
assert int(math.sqrt(big)) != math.isqrt(big)

PASS tính chất: kết quả tăng đúng 1 tại mỗi số chính phương, các n khác giữ nguyên
PASS tính chất: d(i) lẻ <=> i là số chính phương (i < 2000)
int(sqrt(10^18 - 1)) = 1000000000 | isqrt = 999999999


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
rnd = random.Random(2026)
for n in range(0, 600):                                  # vét hết n nhỏ
    check(n)
for trial in range(1000):
    check(rnd.randint(0, 3000))
for trial in range(300):                                 # n lớn theo đề, so các cách nhanh (loop O(sqrt n))
    check(rnd.randint(3001, 10 ** 9))
for k in range(1, 200):                                  # quanh các số chính phương: k^2 - 1, k^2, k^2 + 1
    for n in (k * k - 1, k * k, k * k + 1):
        check(n)
for k in (31621, 31622, 31623):                          # quanh căn của 10^9
    for n in (k * k - 1, k * k, k * k + 1):
        if n <= 10 ** 9:
            check(n)
print("PASS 600 n nhỏ liên tiếp, 1000 ngẫu nhiên nhỏ, 300 ngẫu nhiên lớn, các số quanh số chính phương")

PASS 600 n nhỏ liên tiếp, 1000 ngẫu nhiên nhỏ, 300 ngẫu nhiên lớn, các số quanh số chính phương


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng số thực với `n` quá lớn

In [18]:
def sai_float(n):
    return int(math.sqrt(n))                              # đúng với n <= 10^9, sai với n rất lớn


print("n = 10^9:", sai_float(10 ** 9), "(đúng)", "| n = 10^18 - 1:", sai_float(10 ** 18 - 1), "(sai, đúng: 999999999)")
assert sai_float(10 ** 9) == 31622
assert sai_float(10 ** 18 - 1) != 999999999

n = 10^9: 31622 (đúng) | n = 10^18 - 1: 1000000000 (sai, đúng: 999999999)


**Vì sao sai:** số thực 64 bit chỉ có 53 bit trị số, nên `10¹⁸ − 1` bị làm tròn thành `10¹⁸` trước khi lấy căn. Với đề này (`n ≤ 10⁹`) `int(sqrt(n))` vẫn đúng, nhưng thói quen an toàn là `math.isqrt`.

### Lỗi 2: Quên vòng 1 (hoặc xem bóng ban đầu là sáng/tắt sai)

In [19]:
def sai_quen_vong_1(n):
    on = [False] * (n + 1)
    for k in range(2, n + 1):                              # sai: bắt đầu từ vòng 2 trong khi bóng ban đầu tắt
        for i in range(k, n + 1, k):
            on[i] = not on[i]
    return sum(on)


print("bỏ vòng 1, n = 3 ->", sai_quen_vong_1(3), "| đúng: 1")
assert sai_quen_vong_1(3) != 1

bỏ vòng 1, n = 3 -> 2 | đúng: 1


**Vì sao sai:** nếu bóng ban đầu **tắt**, vòng 1 chính là thao tác bật tất cả; bỏ vòng 1 làm bóng chưa từng bị bật. (Nếu bỏ vòng 1 thì phải đặt trạng thái ban đầu là sáng.)

### Lỗi 3: Làm tròn lên thay vì làm tròn xuống

In [20]:
def sai_ceil(n):
    return math.ceil(math.sqrt(n))                         # sai: số chính phương <= n là floor(sqrt(n))


print("ceil(sqrt(3)) ->", sai_ceil(3), "| đúng: 1")
assert sai_ceil(3) != 1

ceil(sqrt(3)) -> 2 | đúng: 1


**Vì sao sai:** số chính phương không vượt quá `n` là `1², ..., ⌊√n⌋²`. Làm tròn lên đếm thêm một số chính phương lớn hơn `n`.

### Lỗi 4: Dùng `<` thay cho `<=` khi đếm số chính phương

In [21]:
def sai_nho_hon(n):
    k = 0
    while (k + 1) * (k + 1) < n:                            # sai: phải <=
        k += 1
    return k


print("n = 1 ->", sai_nho_hon(1), "| n = 4 ->", sai_nho_hon(4), "| đúng: 1 và 2")
assert sai_nho_hon(1) != 1 and sai_nho_hon(4) != 2

n = 1 -> 0 | n = 4 -> 1 | đúng: 1 và 2


**Vì sao sai:** khi `n` đúng bằng một số chính phương (`n = 1, 4, 9, ...`) thì bóng `n` cũng sáng, nhưng điều kiện `<` bỏ nó đi. Luôn kiểm tra các `n` là số chính phương và `n ± 1` quanh đó.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Đổi trạng thái theo bội số" → xét **số lần đổi = số ước số**; chẵn/lẻ quyết định kết quả.
- Số ước lẻ ⇔ **số chính phương** (ước đi theo cặp, trừ ước tự ghép).
- Đáp án dạng `⌊√n⌋`: dùng `math.isqrt` (Python) để tránh sai số số thực.
- Brainteaser: viết **mô phỏng nhỏ**, in kết quả với `n = 1..20` rồi tìm quy luật; sau đó **chứng minh** và kiểm chứng bằng mô phỏng.
- Họ hàng: Bulb Switcher II (672), III (1375), IV (1529) — mỗi bài có quan sát riêng.
- Kiểm thử quanh các **số chính phương** (`k² − 1, k², k² + 1`) vì đó là chỗ đáp án nhảy.

## 15. Biến thể của bài

1. **Liệt kê** các bóng đang sáng.
2. **Số lần đổi trạng thái** của bóng `i` (`d(i)`).
3. **Chỉ thực hiện `r` vòng đầu** (`r ≤ n`) rồi đếm bóng sáng.
4. **Bóng sáng thứ `m`** chính là `m²`.
5. **Số `n` nhỏ nhất** để có đúng `m` bóng sáng.

In [22]:
def bulbs_on(n):
    """Biến thể 1."""
    return [r * r for r in range(1, math.isqrt(n) + 1)]


def toggles_of(i):
    """Biến thể 2: số lần bóng i bị đổi."""
    return count_divisors(i)


def bulb_switch_rounds(n, r):
    """Biến thể 3 (mô phỏng): chỉ r vòng đầu."""
    on = [False] * (n + 1)
    for k in range(1, r + 1):
        for i in range(k, n + 1, k):
            on[i] = not on[i]
    return sum(on)


def bulb_switch_rounds_fast(n, r):
    """Biến thể 3 (đếm ước <= r): bóng i sáng sau r vòng <=> số ước của i không vượt quá r là lẻ."""
    total = 0
    for i in range(1, n + 1):
        cnt = sum(1 for d in range(1, min(i, r) + 1) if i % d == 0)
        total += cnt % 2
    return total


def mth_bulb_on(m):
    """Biến thể 4."""
    return m * m


def smallest_n_with_m_on(m):
    """Biến thể 5."""
    return m * m


assert bulbs_on(30) == [1, 4, 9, 16, 25]
assert toggles_of(36) == 9 and toggles_of(12) == 6
assert mth_bulb_on(7) == 49 and smallest_n_with_m_on(5) == 25
for n in range(1, 40):
    assert len(bulbs_on(n)) == bulb_switch_isqrt(n)
    assert bulb_switch_rounds(n, n) == bulb_switch_isqrt(n)
    assert smallest_n_with_m_on(bulb_switch_isqrt(n)) <= n
    for r in range(0, n + 1):
        assert bulb_switch_rounds(n, r) == bulb_switch_rounds_fast(n, r)
for m in range(1, 60):
    assert bulb_switch_isqrt(smallest_n_with_m_on(m)) == m
    assert bulb_switch_isqrt(smallest_n_with_m_on(m) - 1) == m - 1
print("PASS biến thể: liệt kê, số lần đổi, r vòng đầu, bóng thứ m, n nhỏ nhất")

PASS biến thể: liệt kê, số lần đổi, r vòng đầu, bóng thứ m, n nhỏ nhất


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đổi trạng thái từng vòng | `O(n log n)` | `O(n)` | oracle `n` nhỏ |
| 2. Đếm ước | `d(i)` lẻ thì sáng | `O(n√n)` | `O(1)` | oracle thứ hai |
| 3. Vòng lặp `k·k ≤ n` | đếm số chính phương | `O(√n)` | `O(1)` | khi chưa tin công thức |
| 4. `math.isqrt` | `⌊√n⌋` | `O(1)` | `O(1)` | **cách nộp bài** |
| 5. Nhị phân / Newton | tự tính căn nguyên | `O(log n)` | `O(1)` | không có `isqrt` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `bulb_switch_isqrt(8)`, `bulb_switch_isqrt(10)` và `bulb_switch_isqrt(100)`.

**Bài 2 (tính tay).** Với `n = 6`, viết trạng thái 6 bóng sau từng vòng và số bóng sáng cuối.

**Bài 3 (điền TODO).** Hoàn thành `bulb_switch_todo` theo Cách 3 (đếm số chính phương bằng vòng lặp).

In [23]:
def bulb_switch_todo(n):
    # TODO 1: k = 0
    # TODO 2: while (k + 1) * (k + 1) <= n: tăng k
    # TODO 3: trả k
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `bulb_switch_binary(n)` theo Cách 5 (tìm `k` lớn nhất có `k·k ≤ n` bằng tìm kiếm nhị phân), cẩn thận vòng lặp không treo.

**Bài 5 (biến thể / mini challenge).** (a) Viết `bulbs_on_list(n)` liệt kê các bóng sáng. (b) Viết `cube_variant(n)`: nếu vòng `k` đổi trạng thái các bóng **chia hết cho `k²`**... hãy mô phỏng và quan sát quy luật cho `n` nhỏ (không cần công thức).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: chỉ các số chính phương sáng. Bài 2: mô phỏng 6 vòng từ trạng thái tắt.
- **Mức 2 (vừa):** Bài 1: `2`, `3`, `10`. Bài 2: sau vòng 1 `[1,1,1,1,1,1]`; vòng 2 `[1,0,1,0,1,0]`; vòng 3 `[1,0,0,0,1,1]`; vòng 4 `[1,0,0,1,1,1]`; vòng 5 `[1,0,0,1,0,1]`; vòng 6 `[1,0,0,1,0,0]`; đáp án `2`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `lo, hi = 0, n`, `mid = (lo + hi + 1) // 2` (lệch lên để tránh lặp vô hạn), nếu `mid * mid <= n` thì `lo = mid` ngược lại `hi = mid - 1`. Bài 5: mô phỏng đúng như đề với điều kiện `i % (k * k) == 0`, in vài `n` đầu và nhận xét.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert bulb_switch_isqrt(8) == 2 and bulb_switch_isqrt(10) == 3 and bulb_switch_isqrt(100) == 10

# Bài 2
def states_after_rounds(n):
    on = [0] * (n + 1)
    history = []
    for k in range(1, n + 1):
        for i in range(k, n + 1, k):
            on[i] ^= 1
        history.append(on[1:])
    return history


hist = states_after_rounds(6)
assert hist == [[1, 1, 1, 1, 1, 1], [1, 0, 1, 0, 1, 0], [1, 0, 0, 0, 1, 1],
                [1, 0, 0, 1, 1, 1], [1, 0, 0, 1, 0, 1], [1, 0, 0, 1, 0, 0]]
assert sum(hist[-1]) == 2

# Bài 3
def bulb_switch_todo_ans(n):
    k = 0
    while (k + 1) * (k + 1) <= n:
        k += 1
    return k


# Bài 4
def bulb_switch_binary(n):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * mid <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo


# Bài 5
def bulbs_on_list(n):
    on = [False] * (n + 1)
    for k in range(1, n + 1):
        for i in range(k, n + 1, k):
            on[i] = not on[i]
    return [i for i in range(1, n + 1) if on[i]]


def cube_variant(n):
    """Vòng k đổi các bóng i chia hết cho k*k (quan sát quy luật cho n nhỏ)."""
    on = [False] * (n + 1)
    for k in range(1, n + 1):
        for i in range(k * k, n + 1, k * k):
            on[i] = not on[i]
    return [i for i in range(1, n + 1) if on[i]]


for n in range(0, 300):
    assert bulb_switch_todo_ans(n) == bulb_switch_binary(n) == bulb_switch_isqrt(n)
    assert bulbs_on_list(n) == bulbs_on(n)
assert cube_variant(40)[:3] == [1, 2, 3]                     # chỉ để xem quy luật; không có công thức bắt buộc
print("quy luật biến thể k*k:", cube_variant(40))
print("PASS đáp án phần 19")

quy luật biến thể k*k: [1, 2, 3, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 19, 21, 22, 23, 26, 29, 30, 31, 32, 33, 34, 35, 37, 38, 39]
PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template**

```python
import math

class Solution:
    def bulbSwitch(self, n: int) -> int:
        return math.isqrt(n)
```

**Checklist**

- [ ] Hiểu bóng `i` bị đổi `d(i)` lần và sáng ⇔ `d(i)` lẻ?
- [ ] Biết `d(i)` lẻ ⇔ `i` là số chính phương (ước đi theo cặp)?
- [ ] Đáp án là `⌊√n⌋` (làm tròn **xuống**), `n = 0` cho `0`?
- [ ] Dùng `math.isqrt` (hoặc nhị phân/Newton), tránh `sqrt` dấu phẩy động với `n` rất lớn?
- [ ] Đã thử `n = 0, 1, 2, 3, 4`, các số chính phương `k²` và `k² ± 1`, `n = 10⁹`?
- [ ] Đã đối chiếu với mô phỏng trực tiếp cho `n` nhỏ?
- [ ] Skeleton đúng `class Solution` / `bulbSwitch(self, n: int) -> int`?